In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = '0'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.5'

import numpy as np

import matplotlib.pyplot as plt

import torch
torch.serialization.add_safe_globals([slice])  # allow this builtin during torch.load

from e3nn import o3


In [ ]:
r_cut = 10.0
num_bessel = 4
max_ell = 2
p_order = 6

test_vec = torch.tensor([1.0, 2.0, 3.0])

# Rotation matrix: 90° about z-axis
Rz_90 = torch.tensor([
    [0.0, -1.0, 0.0],
    [1.0,  0.0, 0.0],
    [0.0,  0.0, 1.0]
])

Rz_180 = torch.tensor([
    [-1.0,  0.0, 0.0],
    [ 0.0, -1.0, 0.0],
    [ 0.0,  0.0, 1.0]
])


test_vec_rot90 = Rz_90 @ test_vec
test_vec_rot180 = Rz_180 @ test_vec

test_vec, test_vec_rot90, test_vec_rot180

### RBF

In [ ]:
bessel_fn = BesselBasis(r_max=r_cut, num_basis=num_bessel)

In [ ]:
r_values = np.linspace(0, r_cut, 100)

# Plot each Bessel basis function
for i in range(num_bessel):
    # Compute the Bessel basis function values for each r
    bessel_values = bessel_fn(torch.tensor(r_values).unsqueeze(-1))[:, i].numpy()
    # Plot the Bessel basis function
    plt.plot(r_values, bessel_values, label=f'Bessel {i+1}')

# Add labels and legend
plt.xlabel('r')
plt.ylabel('Bessel Basis Value')
plt.title('Bessel Basis Functions')
plt.legend()
plt.show()

### Spherical harmonics

In [ ]:
sh_irreps = o3.Irreps.spherical_harmonics(max_ell)

spherical_harmonics = o3.SphericalHarmonics(
    sh_irreps, normalize=True, normalization="component"
)

# spherical harmonics takes a tensor of shape [..., 3] (vectors)
# and returns a tensor of shape [..., irreps_out]

In [ ]:
spherical_harmonics(test_vec_rot180)

### Cutoff fn

In [ ]:
class PolynomialCutoff(torch.nn.Module):
    """Polynomial cutoff function that goes from 1 to 0 as x goes from 0 to r_max.
    Equation (8) -- TODO: from where?
    """

    p: torch.Tensor
    r_max: torch.Tensor

    def __init__(self, r_max: float, p=6):
        super().__init__()
        self.register_buffer("p", torch.tensor(p, dtype=torch.int))
        self.register_buffer(
            "r_max", torch.tensor(r_max, dtype=torch.get_default_dtype())
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.calculate_envelope(x, self.r_max, self.p.to(torch.int))

    @staticmethod
    def calculate_envelope(
        x: torch.Tensor, r_max: torch.Tensor, p: torch.Tensor
    ) -> torch.Tensor:
        r_over_r_max = x / r_max
        envelope = (
            1.0
            - ((p + 1.0) * (p + 2.0) / 2.0) * torch.pow(r_over_r_max, p)
            + p * (p + 2.0) * torch.pow(r_over_r_max, p + 1)
            - (p * (p + 1.0) / 2) * torch.pow(r_over_r_max, p + 2)
        )
        return envelope * (x < r_max)

    def __repr__(self):
        return f"{self.__class__.__name__}(p={self.p}, r_max={self.r_max})"


In [ ]:
cutoff_fn = PolynomialCutoff(r_max=r_cut, p=p_order)
test_distances = torch.linspace(0, r_cut+2, steps=100, dtype=torch.float32)
cutoff_values = cutoff_fn(test_distances)

# Plot the results
plt.figure(figsize=(4, 2))
plt.plot(test_distances.numpy(), cutoff_values.numpy(), label='Cutoff Function')
plt.xlabel('Distance')
plt.ylabel('Cutoff Value')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
tp = o3.FullTensorProduct(
    irreps_in1='2x0e + 3x1o',
    irreps_in2='5x0e + 7x1e'
)
print(tp)
tp.visualize();